# 8.1 Introduction to World Models

> **Environment Setup**: This section is purely theoretical with code demonstrations. No AirSim simulator is needed — only PyTorch.
>
> **AirSim Configuration**: Later experiments (Sections 8.3–8.6) require AirSim. Copy the `settings.json` file from this directory to `~/Documents/AirSim/settings.json`, then launch AirSim. The configuration uses a single drone (Drone1) with 64x64 camera resolution (sized for world model input).
>
> **Python Environment**: Use the `conda activate drone` environment. Install dependencies with `pip install torch gymnasium`.

## Learning Objectives

- Understand the evolution from reactive decision-making to predictive planning
- Master the core world model formula: S + A → S'
- Distinguish between model-based and model-free reinforcement learning
- Understand why latent space modeling is necessary
- Implement a minimal world model from scratch

## 8.1.1 Why Do We Need World Models?

Imagine you're driving. When you see a traffic light turning yellow ahead, you **anticipate**:
- If I accelerate, can I make it through before it turns red?
- If I brake, will the car behind me rear-end me?

You don't actually try either option — instead, you **simulate** the consequences of different actions in your mind, then choose the best course of action.

This is the core idea behind world models: **Build a mental simulator of the environment to predict the outcomes of different actions, enabling better decisions.**

### Limitations of Traditional Approaches

| Approach | Decision Method | Analogy |
|----------|----------------|---------|
| Vision-Language Navigation (VLN) | React to what you see, no future prediction | Walking while only looking at your feet |
| Model-free RL | Trial and error, accumulate experience | Falling 100 times before learning to ride a bike |
| **World Model** | **Simulate in your mind first, then act** | **Imagine riding the bike, fall fewer times** |

For drones, "trial and error" is extremely costly — every crash means hardware damage. World models let drones experiment in a "virtual imagination," dramatically reducing the cost of real-world exploration.

![Evolution from VLN to World Models](figures/wm_evolution.png)

*Figure 8-1: From reactive responses to predictive planning — world models give agents the ability to "anticipate the future"*

## 8.1.2 The Core Formula of World Models

A world model answers one fundamental question:

> **If I am currently in state S and take action A, what will the next state S' be?**

Expressed mathematically:

$$S_{t+1}, R_t = f(S_t, A_t)$$

Where:
- $S_t$: Current state (drone's position, velocity, attitude, etc.)
- $A_t$: Current action (throttle, pitch, roll, and other control inputs)
- $S_{t+1}$: Predicted next state
- $R_t$: Predicted reward (are we closer to the goal? Or did we hit a wall?)
- $f$: The world model (a neural network)

### Concrete Example for Drones

| Input | Meaning | Example |
|-------|---------|---------|
| $S_t$ | Current state | Position(0,0,-5), Velocity(1,0,0), Attitude(level) |
| $A_t$ | Action taken | Tilt left 30 degrees |
| $S_{t+1}$ | Predicted next state | Position(-0.5,0,-5), Velocity(0.5,-1,0), Attitude(left tilt) |
| $R_t$ | Predicted reward | +0.8 (still near the target) |

![World Model Core Formula](figures/wm_core_formula.png)

*Figure 8-2: The core question of world models — given the current state and action, predict the next state and reward*

## 8.1.3 Model-Based RL vs Model-Free RL

| Comparison | Model-Free RL | Model-Based RL |
|------------|---------------|----------------|
| Representative algorithms | DQN, PPO, A2C | DreamerV3, World Models |
| How it works | Trial and error in the real environment | Learn a world model first, then plan within it |
| Sample efficiency | Low (requires massive interaction data) | High (can trial-and-error infinitely in the virtual world) |
| Training cost | High (every step requires real interaction) | Low (most training happens in imagination) |
| Risk | High (real trial-and-error can damage equipment) | Low (virtual trial-and-error is free) |
| Drawback | Simple and direct, no modeling needed | Risk of model bias |

**Model Bias** is the primary risk of model-based RL: if the world model is inaccurate, the good policies the agent discovers in "imagination" may completely fail in the real world. It's like someone who learned to fly in a dream, only to wake up and find they can't actually fly.

## 8.1.4 Latent Space Modeling: Why Not Predict Pixels Directly?

A natural idea is to have the world model directly predict every pixel of the next frame. But this has three serious problems:

**1. Curse of Dimensionality**

A 64x64 RGB image has 64x64x3 = 12,288 pixel values. Predicting this many values is far harder than predicting 6 state variables (position xyz + velocity xyz).

**2. Information Redundancy**

Many pixels in an image are irrelevant to decision-making — the color of the sky, ground textures, distant clouds — none of these affect the drone's flight decisions.

**3. Noise Sensitivity**

Minor pixel-level changes from lighting variations or sensor noise can severely interfere with predictions.

### Solution: Latent Space

```
High-dimensional image (12288-dim) → [Encoder] → Latent state (32-dim) → [World Model] → Predicted latent state → [Decoder] → Predicted image
```

The encoder compresses images into a low-dimensional "latent state vector," retaining only the core information relevant to decision-making. The world model operates in this low-dimensional space, improving efficiency by orders of magnitude.

## 8.1.5 Hands-On: Building the Simplest World Model

Below we implement the simplest possible world model using PyTorch: an MLP (multi-layer perceptron) that takes the current state and action as input, and predicts the next state and reward.

While the real DreamerV3 is far more complex, the core idea is exactly the same: **learn the mapping S + A → S'**.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

# Define the simplest world model
class SimpleWorldModel(nn.Module):
    def __init__(self, state_dim=6, action_dim=4, hidden_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.state_head = nn.Linear(hidden_dim, state_dim)
        self.reward_head = nn.Linear(hidden_dim, 1)

    def forward(self, state, action):
        x = torch.cat([state, action], dim=-1)
        h = self.net(x)
        return self.state_head(h), self.reward_head(h)

model = SimpleWorldModel()
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Input: state(6-dim) + action(4-dim) = 10-dim")
print(f"Output: next_state(6-dim) + reward(1-dim) = 7-dim")

### Generate Simulated Data and Train

We use a simple physics rule to generate training data: a drone moving in a 2D plane, with actions controlling acceleration.

In [ ]:
# Generate simulated data: simple 2D motion
def generate_data(n_episodes=200, steps_per_episode=50):
    states, actions, next_states, rewards = [], [], [], []
    for _ in range(n_episodes):
        pos = np.random.randn(3) * 2  # Initial position
        vel = np.zeros(3)              # Initial velocity
        for _ in range(steps_per_episode):
            state = np.concatenate([pos, vel]).astype(np.float32)
            action = np.random.randn(4).astype(np.float32) * 0.5
            # Simple physics: first 3 dims of action control acceleration, 4th controls vertical thrust
            acc = action[:3] * 0.1
            acc[2] += action[3] * 0.05 - 0.02  # Gravity compensation
            vel = vel * 0.95 + acc  # Damping
            pos = pos + vel * 0.1
            next_state = np.concatenate([pos, vel]).astype(np.float32)
            reward = max(0, 1.0 - np.linalg.norm(pos) / 5.0)  # Higher reward near origin
            states.append(state)
            actions.append(action)
            next_states.append(next_state)
            rewards.append([reward])
    return (torch.tensor(np.array(x)) for x in [states, actions, next_states, rewards])

states, actions, next_states, rewards = generate_data()
print(f"Training data: {states.shape[0]} samples")
print(f"State dimension: {states.shape[1]}, Action dimension: {actions.shape[1]}")

In [ ]:
# Train the world model
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []

for epoch in range(100):
    pred_states, pred_rewards = model(states, actions)
    loss_state = nn.MSELoss()(pred_states, next_states)
    loss_reward = nn.MSELoss()(pred_rewards, rewards)
    loss = loss_state + loss_reward

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

plt.figure(figsize=(8, 3))
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('World Model Training Loss')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Final loss: {losses[-1]:.6f}")

### Test: Using the World Model to "Imagine" Future Trajectories

![Hover vs Obstacle Avoidance Tasks](figures/hover_vs_avoidance.png)

*Figure 8-3: Comparison of the two upcoming experiments — from simple hover control to complex obstacle avoidance decisions*

In [ ]:
# Use the trained world model to predict future trajectories
model.eval()
with torch.no_grad():
    # Initial state: position(2,2,-3), velocity(0,0,0)
    state = torch.tensor([[2.0, 2.0, -3.0, 0.0, 0.0, 0.0]])
    trajectory = [state.numpy()[0, :3]]

    for step in range(50):
        # Simple policy: apply force toward the origin
        pos = state[0, :3]
        action = -pos * 0.3  # Toward origin
        action = torch.cat([action, torch.tensor([0.2])]).unsqueeze(0)  # Add vertical thrust
        state, reward = model(state, action)
        trajectory.append(state.numpy()[0, :3])

trajectory = np.array(trajectory)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# XY plane trajectory
axes[0].plot(trajectory[:, 0], trajectory[:, 1], 'b-o', markersize=3)
axes[0].plot(trajectory[0, 0], trajectory[0, 1], 'go', markersize=10, label='Start')
axes[0].plot(0, 0, 'r*', markersize=15, label='Target (origin)')
axes[0].set_xlabel('X'); axes[0].set_ylabel('Y')
axes[0].set_title('World Model "Imagined" Flight Trajectory (XY Plane)')
axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_aspect('equal')

# Altitude change
axes[1].plot(trajectory[:, 2], 'r-o', markersize=3)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3, label='Ground')
axes[1].set_xlabel('Time Step'); axes[1].set_ylabel('Z (Altitude)')
axes[1].set_title('Altitude Change'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Start: ({trajectory[0,0]:.1f}, {trajectory[0,1]:.1f}, {trajectory[0,2]:.1f})")
print(f"End: ({trajectory[-1,0]:.1f}, {trajectory[-1,1]:.1f}, {trajectory[-1,2]:.1f})")

## 8.1.6 Summary

Key takeaways from this section:

1. **World models** let agents simulate the environment in their "mind," predicting the consequences of different actions
2. Core formula: $S_{t+1}, R_t = f(S_t, A_t)$
3. **Model-based RL** is far more sample-efficient than model-free methods, making it especially suitable for drones where trial-and-error is costly
4. **Latent space modeling** compresses high-dimensional images into low-dimensional vectors, making prediction efficient
5. Even the simplest MLP world model can learn to predict physical motion patterns

In the next section, we'll dive deep into DreamerV3 — one of the most advanced world model architectures available today.